# 🔄 Flussi di Lavoro Base per Agenti con Modelli GitHub (Python)

## 📋 Tutorial di Orchestrazione del Flusso di Lavoro

Questo notebook introduce le potenti capacità del **Workflow Builder** del Microsoft Agent Framework. Impara come creare flussi di lavoro sofisticati e multi-step in grado di gestire processi aziendali complessi e coordinare senza problemi più operazioni di AI.

## 🎯 Obiettivi di Apprendimento

### 🏗️ **Architettura del Flusso di Lavoro**
- **Workflow Builder**: Progettare e orchestrare processi complessi multi-step
- **Esecuzione Basata su Eventi**: Gestire eventi del flusso di lavoro e transizioni di stato
- **Progettazione Visiva del Flusso di Lavoro**: Creare e visualizzare strutture di flussi di lavoro
- **Integrazione Modelli GitHub**: Sfruttare modelli AI all'interno dei contesti di flusso di lavoro

### 🔄 **Orchestrazione del Processo**
- **Operazioni Sequenziali**: Collegare più task dell'agente in ordine logico
- **Logica Condizionale**: Implementare punti decisionali e flussi di lavoro ramificati
- **Gestione degli Errori**: Ripristino robusto degli errori e resilienza del flusso di lavoro
- **Gestione dello Stato**: Tracciare e gestire lo stato di esecuzione del flusso di lavoro

### 📊 **Pattern di Flusso di Lavoro Aziendali**
- **Automazione dei Processi Aziendali**: Automatizzare flussi di lavoro organizzativi complessi
- **Coordinamento Multi-Agente**: Coordinare più agenti specializzati
- **Esecuzione Scalabile**: Progettare flussi di lavoro per operazioni su scala aziendale
- **Monitoraggio & Osservabilità**: Tracciare performance e risultati del flusso di lavoro

## ⚙️ Prerequisiti & Configurazione

### 📦 **Dipendenze Richieste**

Installa il Agent Framework con capacità di workflow:

```bash
pip install agent-framework-core -U
```

### 🔑 **Configurazione Modelli GitHub**

**Configurazione Ambiente (file .env):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

### 🏢 **Casi d'Uso Aziendali**

**Esempi di Processi Aziendali:**
- **Onboarding Clienti**: Flussi multi-step di verifica e setup
- **Pipeline di Contenuti**: Creazione, revisione e pubblicazione automatizzata dei contenuti
- **Elaborazione Dati**: Flussi ETL con trasformazione potenziata da AI
- **Assicurazione della Qualità**: Processi automatizzati di test e validazione

**Benefici del Flusso di Lavoro:**
- 🎯 **Affidabilità**: Esecuzione deterministica con recupero dagli errori
- 📈 **Scalabilità**: Gestione di automazioni di processo ad alto volume
- 🔍 **Osservabilità**: Tracciabilità completa e monitoraggio
- 🔧 **Manutenibilità**: Progettazione visiva e componenti modulari

## 🎨 Pattern di Progettazione del Flusso di Lavoro

### Struttura Base del Flusso di Lavoro
```mermaid
graph TD
    A[Inizio] --> B[Compito Agente 1]
    B --> C{Punto di Decisione}
    C -->|Successo| D[Compito Agente 2]
    C -->|Fallimento| E[Gestore Errori]
    D --> F[Fine]
    E --> F
```

**Componenti Chiave:**
- **WorkflowBuilder**: Motore principale di orchestrazione
- **WorkflowEvent**: Gestione eventi e comunicazione
- **WorkflowViz**: Rappresentazione visiva del flusso di lavoro e debugging

Costruiamo il tuo primo flusso di lavoro intelligente! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load GitHub Models API credentials from .env file
load_dotenv()

In [ ]:
# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = await provider.create_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = await provider.create_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)

In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
